In [1]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


In [2]:
Data_Path = "../data/train"
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 15
NUM_CLASSES = 5   # Oval, Square, Heart, Round, Oblong
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
import os 
print(os.getcwd())
print(os.listdir(Data_Path))

/Users/mahshidsmac/Desktop/Third year/FYP/smart_mirror_application/model/notebook
['Oblong', 'Heart', 'Square', 'Oval', 'Round']


In [4]:
mp_face_mesh = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True
)

def extract_landmarks(image):
    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = mp_face_mesh.process(img_rgb)
    if not result.multi_face_landmarks:
        return None
    return np.array([[lm.x, lm.y] for lm in result.multi_face_landmarks[0].landmark])


I0000 00:00:1767920668.256038 38786282 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [5]:
import math

def dist(a, b):
    return math.dist(a, b)

def geometry_features(landmarks):
    jaw = dist(landmarks[234], landmarks[454])
    height = dist(landmarks[10], landmarks[152])
    cheek = dist(landmarks[93], landmarks[323])
    forehead = dist(landmarks[127], landmarks[356])

    return np.array([
        jaw / height,
        cheek / height,
        forehead / height
    ], dtype=np.float32)


W0000 00:00:1767920668.272129 38786618 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
class FaceShapeDataset(Dataset):
    def __init__(self, samples, labels, transform=None):
        self.samples = samples
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        label = self.labels[idx]

        image = cv2.imread(img_path)

        if image is None:
            raise ValueError(f"Image not found or unable to read: {img_path}")
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

        landmarks = extract_landmarks(image)
        geom = geometry_features(landmarks)

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(geom), label


W0000 00:00:1767920668.280667 38786621 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [7]:
# samples, labels = [], []
# class_map = {name: i for i, name in enumerate(os.listdir(Data_Path))
#              if os.path.isdir(os.path.join(Data_Path, name))}

# for cls in class_map:
#     cls_path = os.path.join(Data_Path, cls)
#     for img in os.listdir(cls_path):
#         img_path = os.path.join(cls_path, img)

#         if os.path.isfile(img_path):
#             samples.append(img_path)
#             labels.append(class_map[cls])
#         # samples.append(os.path.join(Data_Path, cls, img))
#         # labels.append(class_map[cls])

# X_train, X_test, y_train, y_test = train_test_split(
#     samples, labels, test_size=0.2, stratify=labels
# )


samples, labels = [], []

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")

class_names = sorted([
    d for d in os.listdir(Data_Path)
    if os.path.isdir(os.path.join(Data_Path, d)) and not d.startswith('.')
])

class_map = {name: i for i, name in enumerate(class_names)}

print("Class map:", class_map)  # DEBUG

for cls in class_names:
    cls_path = os.path.join(Data_Path, cls)
    for img in os.listdir(cls_path):
        if not img.lower().endswith(VALID_EXTENSIONS):
            continue
        img_path = os.path.join(cls_path, img)
        if os.path.isfile(img_path):
            samples.append(img_path)
            labels.append(class_map[cls])

X_train, X_test, y_train, y_test = train_test_split(
    samples, labels, test_size=0.2, stratify=labels
)


Class map: {'Heart': 0, 'Oblong': 1, 'Oval': 2, 'Round': 3, 'Square': 4}


In [8]:
# Debugging outputs
print("Unique labels:", sorted(set(labels)))
print("Min label:", min(labels))
print("Max label:", max(labels))


Unique labels: [0, 1, 2, 3, 4]
Min label: 0
Max label: 4


In [9]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [10]:
class CNNBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, 256)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [ ]:
class Hybrid_Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = CNNBackbone()
        self.geom_fc = nn.Linear(3, 64)

        self.classifier = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, NUM_CLASSES)
        )

    def forward(self, image, geom):
        cnn_feat = self.cnn(image)
        geom_feat = self.geom_fc(geom)
        fused = torch.cat((cnn_feat, geom_feat), dim=1)
        return self.classifier(fused)


In [12]:
# Debugging outputs

print("NUM_CLASSES:", NUM_CLASSES)
print("Unique train labels:", sorted(set(y_train)))
assert min(y_train) == 0
assert max(y_train) == NUM_CLASSES - 1


NUM_CLASSES: 5
Unique train labels: [0, 1, 2, 3, 4]


In [ ]:
model = Hybrid_Model().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

train_ds = FaceShapeDataset(X_train, y_train, transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for imgs, geoms, labels in train_loader:
        imgs = imgs.to(DEVICE)
        geoms = geoms.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs, geoms)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")


/Users/mahshidsmac/Desktop/Third year/FYP/smart_mirror_application/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/mahshidsmac/Desktop/Third year/FYP/smart_mirror_application/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/Users/mahshidsmac/Desktop/Third year/FYP/smart_mirror_application/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. 

Epoch 1: Loss = 256.0683
Epoch 2: Loss = 132.0008
Epoch 3: Loss = 59.6018
Epoch 4: Loss = 31.1383
Epoch 5: Loss = 25.4608
Epoch 6: Loss = 20.9970
Epoch 7: Loss = 19.8446
Epoch 8: Loss = 18.5479
Epoch 9: Loss = 18.0912
Epoch 10: Loss = 8.4766
Epoch 11: Loss = 11.2430
Epoch 12: Loss = 14.9879
Epoch 13: Loss = 9.6222
Epoch 14: Loss = 8.7632
Epoch 15: Loss = 8.7188


In [14]:
test_ds = FaceShapeDataset(X_test, y_test, transform)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

model.eval()
preds, gts = [], []

with torch.no_grad():
    for imgs, geoms, labels in test_loader:
        outputs = model(imgs.to(DEVICE), geoms.to(DEVICE))
        preds.extend(torch.argmax(outputs, 1).cpu().numpy())
        gts.extend(labels.numpy())

print(classification_report(gts, preds))


              precision    recall  f1-score   support

           0       0.77      0.64      0.69       159
           1       0.54      0.94      0.69       160
           2       0.76      0.56      0.64       160
           3       0.81      0.68      0.74       160
           4       0.88      0.76      0.82       160

    accuracy                           0.71       799
   macro avg       0.75      0.71      0.72       799
weighted avg       0.75      0.71      0.72       799



In [17]:
import json

os.makedirs("../models", exist_ok=True)

# Save model weights
torch.save(model.state_dict(), "../models/face_shape.pt")

# Save class names
with open("../models/class_names.json", "w") as f:
    json.dump(class_names, f)

print("✅ Model and class names saved successfully")


✅ Model and class names saved successfully
